# 0 から作る世界モデル (2) SO-101 のカメラ映像

SO-101 で撮った LeRobot のデータセット (カメラ映像 + 関節角) から、
**「この関節角へ動かしたら、カメラにどう映るか」**を予測する小さな世界モデルを 0 から学習します。
形は Mesa 版と同じ DINO-WM 型です。

```
カメラ画像 ──DINOv2 (凍結)──> 特徴 z_t ─┐
                                        ├─ 世界モデル ─> 次の特徴 z_{t+1}   ← 順動力学 = シミュレータ
関節の目標角 a_t (6 軸 × 3 コマ) ───────┘
(z_t, z_{t+1}) ─ 逆動力学 ─> 関節角                                          ← 映像 → 座標
ゴールの画像 ─ 想像の中で関節角の並びを試す (CEM) ─> 動かし方                ← 計画
```

Mesa 版と違って本物の環境 (実機) を Colab から動かせないので、評価は**学習に使っていないエピソード**の映像で行います。

- ランタイム: **A100** 推奨 (L4 でも動くが、学習に時間がかかる)
- データ: Hugging Face の `oggata/record-stack-blocks` (公開済み) を読み込みます

In [ ]:
!apt-get -qq install -y fonts-noto-cjk > /dev/null
!pip -q install av
import os, math, random, time, json, glob
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

# 日本語の見出しを出すためのフォント (Colab は apt で入れた Noto、Mac はヒラギノ)
import matplotlib.font_manager as fm
for f in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"):
    fm.fontManager.addfont(f)
_have = {f.name for f in fm.fontManager.ttflist}
plt.rcParams["font.family"] = [f for f in ("Noto Sans CJK JP", "Hiragino Sans") if f in _have] + ["DejaVu Sans"]

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
QUICK = os.environ.get("WM_QUICK") == "1"   # 手元の動作確認用。Colab では False のまま
AMP = dict(device_type="cuda", dtype=torch.bfloat16, enabled=DEVICE == "cuda")
torch.manual_seed(0); np.random.seed(0); random.seed(0)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

In [ ]:
# 同じカメラ配置で撮ったデータセットなら、複数並べると学習データが増える
REPOS = ["oggata/record-stack-blocks"]   # 例: ["oggata/record-stack-blocks", "oggata/record-transport-markers"]
CAM = "observation.images.front"
STRIDE = 3                   # 30fps → 10Hz (3 コマに 1 枚)。行動は間の 3 コマ分をまとめて渡す
TOK_GRID = 16                # 16×16 トークン (8 にすると 4 倍速いが細部が粗くなる)
N_TOK = TOK_GRID ** 2
THUMB = TOK_GRID * 8         # 目で見る用の縮小画像 (128×128)
HIST = 3
N_VAL_EPS = 3                # 各データセットの最後の 3 エピソードは評価専用
JOINTS = ["shoulder_pan", "shoulder_lift", "elbow_flex", "wrist_flex", "wrist_roll", "gripper"]
MAX_EPS = 4 if QUICK else None
WM_STEPS = 40 if QUICK else 10000
DEC_STEPS = 40 if QUICK else 3000
BATCH = 8 if QUICK else 32

## 共通の部品 (DINOv2・世界モデル・復元器・逆動力学)

In [ ]:
# 凍結した DINOv2 (vits14)。224×224 の画像 → 16×16 のパッチ特徴 (各 384 次元)。
# 学習はしない。世界モデルはこの特徴の空間で「次に何が見えるか」を予測する。
dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14").to(DEVICE).eval()
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
IMNET_STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
FEAT_DIM = 384

@torch.no_grad()
def dino_encode(imgs, grid):
    """imgs: (B,3,224,224) float [0,1] → (B, grid*grid, 384)。grid<16 なら平均プーリングで粗くする"""
    with torch.autocast(**AMP):
        p = dino.forward_features((imgs - IMNET_MEAN) / IMNET_STD)["x_norm_patchtokens"]
    B = p.shape[0]
    p = p.float().transpose(1, 2).reshape(B, FEAT_DIM, 16, 16)
    if grid != 16:
        p = F.adaptive_avg_pool2d(p, grid)
    return p.flatten(2).transpose(1, 2)

In [ ]:
class WorldModel(nn.Module):
    """順動力学: 「過去 T フレームの特徴 + 各フレームでの行動」→「それぞれの次のフレームの特徴」。

    1 フレーム = N 個のパッチトークン。行動の埋め込みを全トークンに足し、フレーム単位の因果マスクで
    「フレーム i はフレーム 0..i だけを見る」ようにする (DINO-WM と同じ形)。
    出力は「前のフレームからの差分」。最初は差分 0 = 「何も変わらない」と予測する状態から学習が始まる。
    """
    def __init__(self, feat_dim, n_tok, hist, act_dim=None, n_actions=None, d=256, depth=6, heads=8):
        super().__init__()
        self.hist, self.n_tok = hist, n_tok
        self.inp = nn.Linear(feat_dim, d)
        if n_actions is not None:
            self.act = nn.Embedding(n_actions, d)
        else:
            self.act = nn.Sequential(nn.Linear(act_dim, d), nn.GELU(), nn.Linear(d, d))
        self.pos_tok = nn.Parameter(torch.randn(1, 1, n_tok, d) * 0.02)
        self.pos_t = nn.Parameter(torch.randn(1, hist, 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout=0.0, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.out = nn.Linear(d, feat_dim)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)
        f = torch.arange(hist).repeat_interleave(n_tok)
        self.register_buffer("mask", f[None, :] > f[:, None], persistent=False)  # True = 見てはいけない

    def forward(self, z, a):
        # z: (B,T,N,D) / a: (B,T) 行動番号 か (B,T,A) 連続値。位置 t の出力が t+1 の予測
        B, T, N, D = z.shape
        h = self.inp(z) + self.pos_tok + self.pos_t[:, -T:] + self.act(a)[:, :, None, :]
        h = self.blocks(h.reshape(B, T * N, -1), mask=self.mask[:T * N, :T * N])
        return z + self.out(self.norm(h)).reshape(B, T, N, D)


@torch.no_grad()
def imagine(model, z_ctx, a_seq):
    """想像の中で先へ進める (シミュレータとしての使い方)。
    z_ctx: (B,C,N,D) 実際に見た直近 C フレーム / a_seq: (B, C-1+K, ...) 文脈の先頭フレームからの行動
    返り値: (B,K,N,D) 想像した K フレーム先までの特徴。予測を次の入力に使い回すので、先ほど誤差が積もる"""
    H = model.hist
    zs = list(z_ctx.float().unbind(1)); C = len(zs)
    K = a_seq.shape[1] - (C - 1)
    for _ in range(K):
        t = len(zs) - 1; lo = max(0, t - H + 1)
        with torch.autocast(**AMP):
            pred = model(torch.stack(zs[lo:], 1), a_seq[:, lo:t + 1])[:, -1]
        zs.append(pred.float())
    return torch.stack(zs[C:], 1)


class Decoder(nn.Module):
    """特徴 (grid×grid トークン) → 画像 (3, 8·grid, 8·grid)。想像した未来を目で見るためだけに使う。
    世界モデルの学習や計画には一切使わない (画素ではなく特徴で考えるのが DINO-WM の要点)。"""
    def __init__(self, feat_dim, grid, ch=128):
        super().__init__()
        self.grid = grid
        def up(i, o):
            return nn.Sequential(nn.ConvTranspose2d(i, o, 4, 2, 1), nn.GroupNorm(8, o), nn.GELU(),
                                 nn.Conv2d(o, o, 3, 1, 1), nn.GroupNorm(8, o), nn.GELU())
        self.net = nn.Sequential(nn.Conv2d(feat_dim, 2 * ch, 1), up(2 * ch, 2 * ch), up(2 * ch, ch),
                                 up(ch, ch // 2), nn.Conv2d(ch // 2, 3, 3, 1, 1))

    def forward(self, z):
        B, N, D = z.shape
        return torch.sigmoid(self.net(z.transpose(1, 2).reshape(B, D, self.grid, self.grid)))


class InverseDynamics(nn.Module):
    """逆動力学: (今のフレーム, 次のフレーム) の特徴 → その間の行動。「映像 → 行動 (座標)」の部分"""
    def __init__(self, feat_dim, n_tok, out_dim, r=32):
        super().__init__()
        self.red = nn.Linear(feat_dim, r)
        self.mlp = nn.Sequential(nn.Linear(2 * n_tok * r, 512), nn.GELU(),
                                 nn.Linear(512, 512), nn.GELU(), nn.Linear(512, out_dim))

    def forward(self, z0, z1):
        return self.mlp(torch.cat([self.red(z0).flatten(1), self.red(z1).flatten(1)], 1))


def valid_starts(ep_start, ep_len, H):
    """窓 (最大 H+1 フレーム) が 1 つのエピソードに収まる開始位置の一覧"""
    return torch.cat([torch.arange(s, s + n - H) for s, n in zip(ep_start, ep_len) if n > H])


def train_wm(model, feats, acts, starts, steps, batch=64, lr=3e-4, log_every=500):
    """feats: (M,N,D) 全フレームを並べた特徴 / acts: (M,) か (M,A) フレーム i で取った行動 / starts: valid_starts"""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps, pct_start=0.05)
    H = model.hist; losses = []; t0 = time.time()
    model.train()
    for step in range(steps):
        T = random.randint(1, H)                       # 文脈の長さもばらす (始めのうちは過去が短いので)
        i = starts[torch.randint(0, len(starts), (batch,))]
        idx = (i[:, None] + torch.arange(T + 1)).to(feats.device)
        z = feats[idx].float()                          # (B,T+1,N,D)
        a = acts[idx[:, :T]]
        with torch.autocast(**AMP):
            pred = model(z[:, :T], a)
        loss = F.mse_loss(pred.float(), z[:, 1:])
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); losses.append(loss.item())
        if (step + 1) % log_every == 0 or step + 1 == steps:
            print(f"step {step+1:6d}  loss {np.mean(losses[-log_every:]):.4f}  ({time.time()-t0:.0f}s)")
    model.eval()
    return losses


def train_decoder(dec, feats, thumbs, steps, batch=64, lr=1e-3, log_every=500):
    """thumbs: (M,H,W,3) uint8。特徴 → 画像 を学習する"""
    opt = torch.optim.AdamW(dec.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    dec.train(); losses = []
    for step in range(steps):
        i = torch.randint(0, len(feats), (batch,), device=feats.device)
        x = thumbs[i].permute(0, 3, 1, 2).float() / 255
        with torch.autocast(**AMP):
            y = dec(feats[i].float())
        loss = F.l1_loss(y.float(), x)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        losses.append(loss.item())
        if (step + 1) % log_every == 0 or step + 1 == steps:
            print(f"decoder step {step+1:6d}  L1 {np.mean(losses[-log_every:]):.4f}")
    dec.eval()


@torch.no_grad()
def decode(dec, z):
    """(…,N,D) → (…,H,W,3) numpy [0,1]"""
    sh = z.shape[:-2]
    with torch.autocast(**AMP):
        img = dec(z.reshape(-1, *z.shape[-2:]).float()).float()
    return img.permute(0, 2, 3, 1).reshape(*sh, *img.shape[2:], 3).cpu().numpy()

In [ ]:
def show_rows(rows, cols_title=None, size=1.6, title=None):
    """rows: {行の名前: [画像 (H,W,3), ...]} を表にして並べる"""
    names = list(rows); n = max(len(v) for v in rows.values())
    fig, ax = plt.subplots(len(names), n, figsize=(size * n * 1.2, size * len(names)), squeeze=False)
    for r, name in enumerate(names):
        for c in range(n):
            a = ax[r, c]; a.axis("off")
            if c < len(rows[name]):
                a.imshow(np.clip(rows[name][c], 0, 1))
            if c == 0:
                a.text(-0.08, 0.5, name, transform=a.transAxes, ha="right", va="center", fontsize=10)
            if r == 0 and cols_title is not None:
                a.set_title(cols_title[c], fontsize=9)
    if title: fig.suptitle(title)
    plt.tight_layout(); plt.show()


def animate_rows(rows, fps=6, size=2.2):
    """rows: {名前: (T,H,W,3)} を横に並べた動画 (Colab の中で再生できる HTML)"""
    names = list(rows); T = min(len(v) for v in rows.values())
    fig, ax = plt.subplots(1, len(names), figsize=(size * len(names) * 1.2, size + 0.4), squeeze=False)
    ims = []
    for a, name in zip(ax[0], names):
        a.axis("off"); a.set_title(name, fontsize=10)
        ims.append(a.imshow(np.clip(rows[name][0], 0, 1)))
    def upd(t):
        for im, name in zip(ims, names):
            im.set_data(np.clip(rows[name][t], 0, 1))
        return ims
    anim = animation.FuncAnimation(fig, upd, frames=T, interval=1000 / fps)
    plt.close(fig)
    return HTML(anim.to_jshtml())

## データを読む (動画 → 10Hz の画像 → DINOv2 の特徴)

In [ ]:
# ── LeRobot v3.0 のデータセットを読む (lerobot は入れずに、parquet と mp4 を直接読む) ──
import pandas as pd, av
from huggingface_hub import snapshot_download

def read_parquets(pattern):
    files = sorted(f for f in glob.glob(pattern) if not os.path.basename(f).startswith("._"))
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

def decode_episode(path, t_from, t_to, n_frames):
    """mp4 (AV1) から [t_from, t_to) のコマを読み、STRIDE コマおきに返す (k, H, W, 3) uint8"""
    with av.open(path) as c:
        s = c.streams.video[0]; fps = float(s.average_rate)
        c.seek(int(t_from / s.time_base), stream=s, backward=True, any_frame=False)
        out = {}
        for fr in c.decode(s):
            t = float(fr.pts * s.time_base)
            k = int(round((t - t_from) * fps))
            if k < 0: continue
            if k >= n_frames or t >= t_to - 1e-4: break
            if k % STRIDE == 0: out[k] = fr.to_ndarray(format="rgb24")
    ks = sorted(out)
    return np.stack([out[k] for k in ks]), ks

feats_l, thumbs_l, acts_l, states_l, ep_start, ep_len, is_val = [], [], [], [], [], [], []
n = 0; t0 = time.time()
for repo in REPOS:
    root = snapshot_download(repo, repo_type="dataset", allow_patterns=["meta/*", "data/*", f"videos/{CAM}/*"])
    info = json.load(open(f"{root}/meta/info.json"))
    # ★ Hub に上げ直したデータセットには、前のアップロードの古いファイル (番号の大きい file-00N) が残ることがある。
    #   同じエピソード番号の行が 2 組になるので、番号の小さいファイルの行を優先し、info.json のエピソード数で切る。
    #   フレームの行も「その行が指す data ファイルの index の範囲」で引く (エピソード番号で引くと古い行が混ざる)。
    eps = read_parquets(f"{root}/meta/episodes/*/*.parquet").drop_duplicates("episode_index", keep="first")
    eps = eps[eps.episode_index < info["total_episodes"]].sort_values("episode_index")
    if MAX_EPS: eps = eps.head(MAX_EPS)
    data_files = {}
    for j, (_, e) in enumerate(eps.iterrows()):
        dpath = f"{root}/" + info["data_path"].format(chunk_index=e["data/chunk_index"], file_index=e["data/file_index"])
        if dpath not in data_files: data_files[dpath] = pd.read_parquet(dpath)
        d = data_files[dpath]
        r = d[(d["index"] >= e["dataset_from_index"]) & (d["index"] < e["dataset_to_index"])].sort_values("frame_index")
        assert len(r) == e["length"], (repo, e["episode_index"], len(r), e["length"])
        act = np.stack(r["action"].to_numpy()).astype(np.float32)          # (L,6) 関節の目標角 (度)
        st = np.stack(r["observation.state"].to_numpy()).astype(np.float32)
        vpath = f"{root}/" + info["video_path"].format(video_key=CAM, chunk_index=e[f"videos/{CAM}/chunk_index"],
                                                        file_index=e[f"videos/{CAM}/file_index"])
        frames, ks = decode_episode(vpath, e[f"videos/{CAM}/from_timestamp"], e[f"videos/{CAM}/to_timestamp"], len(r))
        # 行動: 各コマ k から次の 1 枚 (k+STRIDE) までの 3 コマ分の目標角を並べる (足りない分は最後を繰り返す)
        idx = np.minimum(np.array(ks)[:, None] + np.arange(STRIDE), len(act) - 1)
        x = torch.from_numpy(frames).to(DEVICE).permute(0, 3, 1, 2).float() / 255
        f, th = [], []
        for c in x.split(64):
            f.append(dino_encode(F.interpolate(c, size=(224, 224), mode="bilinear", antialias=True), TOK_GRID).half())
            th.append((F.interpolate(c, size=(THUMB, THUMB), mode="bilinear", antialias=True).clamp(0, 1) * 255).round().byte().permute(0, 2, 3, 1))
        feats_l.append(torch.cat(f)); thumbs_l.append(torch.cat(th))
        acts_l.append(torch.from_numpy(act[idx].reshape(len(ks), -1)))
        states_l.append(torch.from_numpy(st[ks]))
        ep_start.append(n); ep_len.append(len(ks)); n += len(ks)
        is_val.append(j >= len(eps) - N_VAL_EPS)
    print(f"{repo}: {len(eps)} エピソード ({time.time()-t0:.0f}s)")

feats = torch.cat(feats_l); thumbs = torch.cat(thumbs_l)
acts_raw = torch.cat(acts_l).to(DEVICE); states = torch.cat(states_l).to(DEVICE)
print(f"全部で {len(feats):,} 枚 (10Hz) / 評価用エピソード {sum(is_val)} 本")

# 標準化 (特徴も行動も、学習用エピソードだけで平均と標準偏差を取る)
tr_mask = torch.zeros(len(feats), dtype=torch.bool, device=DEVICE)
for s, l, v in zip(ep_start, ep_len, is_val):
    if not v: tr_mask[s:s + l] = True
FEAT_MU = feats[tr_mask].float().mean((0, 1)); FEAT_SD = feats[tr_mask].float().std((0, 1)) + 1e-4
feats = ((feats.float() - FEAT_MU) / FEAT_SD).half()
ACT_MU = acts_raw[tr_mask].mean(0); ACT_SD = acts_raw[tr_mask].std(0) + 1e-3
acts = (acts_raw - ACT_MU) / ACT_SD
def norm_act(a_deg):  # (…, 6·STRIDE) 度 → 標準化
    return (a_deg - ACT_MU) / ACT_SD

tr_eps = [(s, l) for s, l, v in zip(ep_start, ep_len, is_val) if not v]
va_eps = [(s, l) for s, l, v in zip(ep_start, ep_len, is_val) if v]
show_rows({"カメラ": list(thumbs[torch.linspace(0, len(thumbs) - 1, 6).long()].cpu().numpy() / 255)}, size=2)

## 学習する

In [ ]:
wm = WorldModel(FEAT_DIM, N_TOK, HIST, act_dim=6 * STRIDE).to(DEVICE)
print(f"世界モデル {sum(p.numel() for p in wm.parameters())/1e6:.1f}M パラメータ")
starts = valid_starts(*zip(*tr_eps), HIST)
losses = train_wm(wm, feats, acts, starts, WM_STEPS, batch=BATCH)
plt.figure(figsize=(6, 3)); plt.plot(np.convolve(losses, np.ones(50) / 50, "valid")); plt.yscale("log")
plt.title("世界モデルの学習 (次の特徴の二乗誤差)"); plt.xlabel("step"); plt.show()

In [ ]:
dec = Decoder(FEAT_DIM, TOK_GRID).to(DEVICE)
tr_idx = torch.cat([torch.arange(s, s + l) for s, l in tr_eps]).to(DEVICE)
train_decoder(dec, feats[tr_idx], thumbs[tr_idx], DEC_STEPS)
va_idx = torch.cat([torch.arange(s, s + l) for s, l in va_eps]).to(DEVICE)
i = va_idx[torch.linspace(0, len(va_idx) - 1, 6).long()]
show_rows({"実際": list(thumbs[i].cpu().numpy() / 255), "特徴から復元": list(decode(dec, feats[i]))},
          size=2, title="評価用エピソードで、特徴 → 画像 の復元 (「想像」を見るときの画質の上限)")

## 1. どのくらい先まで当たるか

評価用エピソードで、直近 3 枚と、その先の関節の目標角 (記録どおり) を渡して 1 秒 (10 枚) 先まで想像させます。
比べる相手は「何も変わらないと予想する」素朴なやり方です。

In [ ]:
K = 10
w = torch.tensor([s + int(o) for s, l in va_eps if l > HIST + K
                  for o in np.linspace(0, l - HIST - K - 1, 8)], device=DEVICE)
ctx = feats[w[:, None] + torch.arange(HIST, device=DEVICE)].float()
a_seq = acts[w[:, None] + torch.arange(HIST - 1 + K, device=DEVICE)]
truth = feats[w[:, None] + HIST + torch.arange(K, device=DEVICE)].float()
pred = imagine(wm, ctx, a_seq)
err_wm = ((pred - truth) ** 2).mean((0, 2, 3)).cpu().numpy()
err_copy = ((ctx[:, -1:] - truth) ** 2).mean((0, 2, 3)).cpu().numpy()
plt.figure(figsize=(6, 3.2))
plt.plot(np.arange(1, K + 1) / 10, err_copy, "o-", label="何も変わらないと予想")
plt.plot(np.arange(1, K + 1) / 10, err_wm, "o-", label="世界モデルの想像")
plt.xlabel("何秒先か"); plt.ylabel("特徴の二乗誤差"); plt.legend(); plt.grid(alpha=.3); plt.show()
print("0.1 秒先: 世界モデル %.3f / 何も変わらない %.3f" % (err_wm[0], err_copy[0]))
print("1.0 秒先: 世界モデル %.3f / 何も変わらない %.3f" % (err_wm[-1], err_copy[-1]))

# 動きの大きい場面を選んで画像で見比べる
j = int(((truth[:, -1] - ctx[:, -1]) ** 2).mean((1, 2)).argmax())
steps = [0, 2, 4, 6, 9]
show_rows({"実際": list(thumbs[w[j] + HIST + torch.tensor(steps, device=DEVICE)].cpu().numpy() / 255),
           "想像": list(decode(dec, pred[j, steps]))}, [f"{(s+1)/10:.1f} 秒先" for s in steps], size=2)
display(animate_rows({"実際": thumbs[w[j] + HIST + torch.arange(K, device=DEVICE)].cpu().numpy() / 255,
                      "想像": decode(dec, pred[j])}, fps=5, size=3))

## 2. 「もし別の動かし方をしたら」を想像する

同じ場面から、関節の目標角だけを書き換えて想像させます。記録にない動かし方ほど想像はあやしくなります
(データに無い世界は知らない、という世界モデルの限界がそのまま見えます)。

In [ ]:
K = 10
s0 = w[len(w) // 2].item()
ctx = feats[s0:s0 + HIST].float()[None]
base = acts_raw[s0:s0 + HIST - 1 + K].clone()                      # 記録どおりの目標角 (度)
def edit(joint, delta):
    a = base.clone().view(len(base), STRIDE, 6)
    a[HIST - 1:, :, JOINTS.index(joint)] += delta                   # これから先だけ書き換える
    return a.view(len(base), -1)
VARIANTS = {"記録どおり": base, "根元を +25°": edit("shoulder_pan", 25), "根元を -25°": edit("shoulder_pan", -25),
            "肩を +20° (前へ倒す)": edit("shoulder_lift", 20), "グリッパーを開く": edit("gripper", 30)}
steps = [1, 4, 6, 9]
rows, anim = {}, {}
for name, a in VARIANTS.items():
    imag = imagine(wm, ctx, norm_act(a)[None])[0]
    rows[name] = list(decode(dec, imag[steps])); anim[name] = decode(dec, imag)
show_rows(rows, [f"{(s+1)/10:.1f} 秒先" for s in steps], size=2)
display(animate_rows(anim, fps=5, size=2.4))

## 3. 映像から関節角を当てる (逆動力学)

連続する 2 枚の特徴だけから、その間の関節の目標角を当てます。前回の説明の「映像 → 座標」の部分です。

In [ ]:
inv = InverseDynamics(FEAT_DIM, N_TOK, 6 * STRIDE).to(DEVICE)
opt = torch.optim.AdamW(inv.parameters(), lr=1e-3, weight_decay=0.01)
pairs = valid_starts(*zip(*tr_eps), 1).to(DEVICE)
for step in range(100 if QUICK else 3000):
    i = pairs[torch.randint(0, len(pairs), (128,), device=DEVICE)]
    loss = F.mse_loss(inv(feats[i].float(), feats[i + 1].float()), acts[i])
    opt.zero_grad(); loss.backward(); opt.step()
inv.eval()

vp = valid_starts(*zip(*va_eps), 1).to(DEVICE)
with torch.no_grad():
    guess = torch.cat([inv(feats[c].float(), feats[c + 1].float()) for c in vp.split(512)]) * ACT_SD + ACT_MU
truth_a = acts_raw[vp]
mae = (guess - truth_a).abs().view(-1, STRIDE, 6).mean((0, 1)).cpu().numpy()
mae0 = (ACT_MU - truth_a).abs().view(-1, STRIDE, 6).mean((0, 1)).cpu().numpy()
print("関節ごとの平均誤差 (度)   映像から当てた / 平均値で答えた")
for jn, a, b in zip(JOINTS, mae, mae0):
    print(f"  {jn:14s} {a:6.1f} / {b:6.1f}")

## 4. ゴールの画像から動かし方を計画する

評価用エピソードのうち**アームが大きく動いた場面**で、0.8 秒後の 1 枚を「ゴールの画像」として渡し、その絵になる関節の目標角の並びを、
想像の中で試して探します (CEM)。実際にそこへ動いたときの記録 (本当の目標角) と比べて、
「今の目標角のまま動かない」と答えた場合より近ければ、映像だけから動かし方を逆算できていることになります。

実機では、ここで得た目標角を `move_sequence` に渡せば同じ計画を実行できます (このノートブックでは実機は動かしません)。

In [ ]:
K, POP, ITERS, ELITE = (3, 8, 2, 4) if QUICK else (8, 300, 5, 30)
N_CASES = 2 if QUICK else 30
INIT_STD = torch.tensor([10, 10, 10, 10, 10, 15.0], device=DEVICE)   # 探す幅 (度)

@torch.no_grad()
def cem_joints(z_hist, a_hist_n, a_now_deg, z_goal):
    """各歩の目標角 (6 軸, 3 コマ同じ値) を K 歩分探す。返り値 (K, 6) 度"""
    mean = a_now_deg[None].repeat(K, 1); std = INIT_STD[None].repeat(K, 1)
    for _ in range(ITERS):
        cand = mean + std * torch.randn(POP, K, 6, device=DEVICE)
        a = norm_act(cand.repeat(1, 1, STRIDE))                          # 3 コマとも同じ目標角
        full = torch.cat([a_hist_n[None].expand(POP, -1, -1), a], 1)
        zf = imagine(wm, z_hist[None].expand(POP, -1, -1, -1), full)
        cost = ((zf[:, -1] - z_goal) ** 2).mean((1, 2))
        el = cand[cost.topk(ELITE, largest=False).indices]
        mean, std = el.mean(0), el.std(0) + 0.5
    return mean

# 動きの大きい場面を選ぶ (ほとんど動かない場面だと「動かない」と答えるだけで当たってしまう)
cand = torch.tensor([c for s, l in va_eps for c in range(s, s + l - HIST - K, 2)], device=DEVICE)
motion = (acts_raw[cand + HIST - 1 + K - 1, -6:] - acts_raw[cand + HIST - 1, :6]).abs()[:, :5].sum(1)
cases = cand[motion.argsort(descending=True)][::3][:N_CASES].tolist()   # 近すぎる場面が並ばないよう 3 つおき
err_plan, err_stay = [], []
for c in cases:
    z_hist = feats[c:c + HIST].float()
    a_hist_n = acts[c:c + HIST - 1]
    now = acts_raw[c + HIST - 1].view(STRIDE, 6)[0]                    # 今の目標角
    goal_t = c + HIST - 1 + K
    plan = cem_joints(z_hist, a_hist_n, now, feats[goal_t].float())
    real = acts_raw[goal_t - 1].view(STRIDE, 6)[-1]                    # 本当にゴールの絵になったときの目標角
    err_plan.append((plan[-1] - real).abs().cpu().numpy()); err_stay.append((now - real).abs().cpu().numpy())
err_plan, err_stay = np.mean(err_plan, 0), np.mean(err_stay, 0)
print(f"ゴールの絵になる目標角との平均誤差 (度, {len(cases)} 場面)   計画 / 動かないと答えた")
for jn, a, b in zip(JOINTS, err_plan, err_stay):
    print(f"  {jn:14s} {a:6.1f} / {b:6.1f}")

# 最後の場面: ゴールの絵と、計画した動かし方で想像した絵
a_full = torch.cat([a_hist_n, norm_act(plan.repeat(1, STRIDE))], 0)[None]
imag = imagine(wm, z_hist[None], a_full)[0]
show_rows({"今": [thumbs[c + HIST - 1].cpu().numpy() / 255],
           "ゴール": [thumbs[goal_t].cpu().numpy() / 255],
           "計画どおり動かした想像": [decode(dec, imag[-1:])[0]]}, size=2.4)
print("計画した目標角 (度):")
print(pd.DataFrame(plan.cpu().numpy().round(1), columns=JOINTS).to_string())

## うまくいかないときに触るところ

- **想像がすぐぼやける / 動きを無視する**: データが少ないのが一番の原因です (1 データセット = 約 30 エピソード)。
  同じカメラ配置のデータセットを `REPOS` に足すのが一番効きます
- **学習が遅い**: `TOK_GRID = 8` にする (トークンが 1/4)
- 計画は「ゴールの絵に一番近い想像」を探すだけなので、似た絵が多い場面 (アームが画面外など) では外れます

## 次の一歩

- **世界モデル向けのデータを撮る**: 作業のお手本だけだと「よく使う動き」しか学べません。
  関節をいろいろな向きにランダムに動かした記録 (record_motion や LeRobot の record) を混ぜると、
  「記録にない動かし方」の想像が良くなります
- 計画した目標角を実機の `move_sequence` に渡して、本当にゴールの絵に近づくかを確かめる